# Label spreading di graf tetangga teks (semi-supervised, transduktif)
**Kapan dipakai:** label sedikit tapi teks tanpa label banyak (test, atau file teks tambahan dari
panitia). Label "mengalir" ke tetangga terdekat di graf kemiripan, jadi teks test ikut membentuk
batas kelas. Beda dengan `semi_supervised_teks` (pseudo-label satu model): di sini struktur
kelompok teks yang dipakai.

1. TF-IDF kata + huruf → SVD (`N_COMPONENTS`) → vektor dinormalkan (kosinus).
2. Graf k tetangga (`N_NEIGHBORS`) atas train + test (+ `UNLABELED_CSV`).
3. Per lipatan: label baris validasi disembunyikan, `LabelSpreading` mengisi semuanya.
4. Banding dengan SGD TF-IDF biasa dan rata-rata keduanya.

**Data demo:** emosi tweet Indonesia (5 kelas, 3,5 ribu train). `LABEL_SHARE` < 1 menyembunyikan
sebagian label train untuk meniru "label sedikit".

In [ ]:
import os
import time

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize
from sklearn.semi_supervised import LabelSpreading

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/graf_label_teks"
OOF_FOLDER = "../../../outputs/text/oof"
DATA_FOLDER = "../../../data_latihan/text/classification"    # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/emot_train.csv"                  # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/emot_test.csv"                    # GANTI saat lomba
KEY_CSV = f"{DATA_FOLDER}/emot_test_labels.csv"              # GANTI saat lomba (kunci, kalau ada)
UNLABELED_CSV = None                         # GANTI: teks tambahan tanpa label (kolom TEXT_COLS)

ID_COL = "id"                                # GANTI saat lomba
TARGET = "label"                             # GANTI saat lomba
TEXT_COLS = ["text"]                         # GANTI saat lomba
MODEL_NAME = "graf_label"
LABEL_SHARE = 1.0            # demo: 0.2 = hanya 20% label train dipakai
N_COMPONENTS = 256
N_NEIGHBORS = 10
SPREAD_ALPHA = 0.2           # 0 = label asli dikunci, makin besar makin bebas berubah
N_FOLDS = 5
SEED = 42


def metric(truth, pred):
    return f1_score(truth, pred.argmax(axis=1), average="macro")      # GANTI: metrik lomba


print("uji metrik:", metric(np.array([0, 1]), np.array([[0.9, 0.1], [0.2, 0.8]])))

## Baca data + vektor SVD

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
tables = [train, test]
if UNLABELED_CSV is not None:
    tables.append(pd.read_csv(UNLABELED_CSV))
texts = []
for table in tables:
    text = table[TEXT_COLS].fillna("")
    texts.append(text.astype(str).agg(" . ".join, axis=1))
all_text = pd.concat(texts, ignore_index=True)
CLASSES, y = np.unique(train[TARGET].astype(str), return_inverse=True)
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                             max_features=100000, sublinear_tf=True)
matrix = hstack([word_tfidf.fit_transform(all_text), char_tfidf.fit_transform(all_text)])
matrix = matrix.tocsr()
vectors = normalize(TruncatedSVD(N_COMPONENTS, random_state=SEED).fit_transform(matrix))
n_train = len(train)
n_test = len(test)
print("baris graf:", len(all_text), "| kelas:", list(CLASSES), "| SVD:", vectors.shape)

## Lipatan + label yang dipakai
`known` = baris train yang labelnya boleh dipakai (semua, kecuali demo `LABEL_SHARE` < 1).

In [ ]:
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(train, y))
rng = np.random.default_rng(SEED)
known = rng.random(n_train) < LABEL_SHARE
print("label dipakai:", known.sum(), "dari", n_train)

## Label spreading vs SGD TF-IDF (lipatan sama)

In [ ]:
def spread_fold(idx_fit):
    labels = np.full(len(all_text), -1)
    rows = idx_fit[known[idx_fit]]
    labels[rows] = y[rows]
    spreader = LabelSpreading(kernel="knn", n_neighbors=N_NEIGHBORS, alpha=SPREAD_ALPHA,
                              max_iter=100)
    spreader.fit(vectors, labels)
    return spreader.label_distributions_


def sgd_fold(idx_fit):
    rows = idx_fit[known[idx_fit]]
    fitted = SGDClassifier(loss="log_loss", alpha=1e-5, random_state=SEED)
    fitted.fit(matrix[rows], y[rows])
    proba = np.zeros((len(all_text), len(CLASSES)))
    proba[:, fitted.classes_] = fitted.predict_proba(matrix)
    return proba


RESULTS = {}
for name, runner in [("label spreading", spread_fold), ("SGD TF-IDF", sgd_fold)]:
    START = time.time()
    oof = np.zeros((n_train, len(CLASSES)))
    test_pred = np.zeros((n_test, len(CLASSES)))
    for idx_fit, idx_val in folds:
        proba = runner(idx_fit)
        oof[idx_val] = proba[idx_val]
        test_pred += proba[n_train: n_train + n_test] / N_FOLDS
    RESULTS[name] = (oof, test_pred)
    print(f"{name:16s} | OOF {metric(y, oof):.4f} | detik {time.time() - START:.0f}")

## Rata-rata + simpan
OOF (n × K, kolom = `np.unique(train[TARGET])`) masuk `ensemble_oof` sebagai model ber-sudut
pandang beda (graf), biasanya menambah keragaman ensemble.

In [ ]:
blend_oof = (RESULTS["label spreading"][0] + RESULTS["SGD TF-IDF"][0]) / 2
blend_test = (RESULTS["label spreading"][1] + RESULTS["SGD TF-IDF"][1]) / 2
RESULTS["rata-rata"] = (blend_oof, blend_test)
rows = []
for name in RESULTS:
    rows.append({"cara": name, "OOF": round(metric(y, RESULTS[name][0]), 4)})
table = pd.DataFrame(rows)
table.to_csv(f"{OUTPUT_FOLDER}/banding_metode.csv", index=False)
print(table.to_string(index=False))
spread_oof, spread_test = RESULTS["label spreading"]
np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", spread_oof.astype("float32"))
np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", spread_test.astype("float32"))
log_path = f"{OOF_FOLDER}/percobaan.csv"
log_row = pd.DataFrame([{"waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
                         "notebook": "graf_label_teks", "model": MODEL_NAME, "metrik": "f1_macro",
                         "skor_cv": round(metric(y, spread_oof), 4), "detik": 0,
                         "catatan": f"k {N_NEIGHBORS}, alpha {SPREAD_ALPHA}, label {LABEL_SHARE}"}])
log_row.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)

## Cek dengan kunci jawaban (HAPUS saat lomba)
File kirim: pakai `ensemble_oof` (gabung OOF ini dengan model lain) atau langsung argmax
`RESULTS["rata-rata"][1]` dengan pola file kirim notebook lain.

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].astype(str).values, index=key[ID_COL].values)
truth = np.searchsorted(CLASSES, test[ID_COL].map(key_by_id).to_numpy())
for name in RESULTS:
    print(f"{name:16s} | test {metric(truth, RESULTS[name][1]):.4f}")